**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO *PROTECT SHARED STATE WITH A LOCK AND KEEP A CONCURRENT RESERVATION OPERATION CONSISTENT. 🐍🔒***

One item remains, but two requests arrive together. Checking stock and updating it separately can allow both requests to succeed.

We’ll compare no lock, a lock around only the write, and a lock around the entire check-and-update operation. A controlled thread interleaving makes the bug repeatable. We’ll also verify that a context manager releases the lock after an exception, without rolling back state.

---

### 1. Imports

Python 3 standard library only; no package installation is required. This notebook uses threads in one process, not multiple services or database transactions.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier, Lock
from pathlib import Path
import json
import platform


---

### 2. Compare the lock boundaries

In the two broken cases, a barrier forces both threads to read `1` before either writes. Locking only the later write cannot repair that stale decision. In the correct case, the start barrier is **outside** the lock; each worker reads, checks and updates while holding the same shared lock.

The barrier is test instrumentation, not production reservation logic. Placing a two-worker barrier inside a lock could deadlock: the first thread would wait for a second thread that cannot enter. No sleeps or assumptions about thread scheduling are needed.

In [2]:
def run_case(mode):
    if mode not in {"unlocked", "write_only", "whole_operation"}:
        raise ValueError("Unknown mode")
    stock = 1
    stock_lock = Lock()  # One shared instance for both workers.
    rendezvous = Barrier(2, timeout=5)

    def reserve():
        nonlocal stock
        if mode == "whole_operation":
            rendezvous.wait()  # Test start gate; outside the critical section.
            with stock_lock:
                if stock == 0:
                    return False
                stock -= 1
                return True

        snapshot = stock
        rendezvous.wait()  # Force both broken workers to read the same value.
        if snapshot == 0:
            return False
        if mode == "write_only":
            with stock_lock:
                stock = snapshot - 1  # The decision still used a stale read.
        else:
            stock = snapshot - 1
        return True

    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(reserve) for _ in range(2)]
        accepted = [future.result(timeout=10) for future in futures]
    return {"accepted": sum(accepted), "rejected": 2 - sum(accepted),
            "remaining": stock, "outcomes": accepted,
            "invariant_ok": sum(accepted) + stock == 1}

results = {mode: run_case(mode)
           for mode in ("unlocked", "write_only", "whole_operation")}
for mode, row in results.items():
    print(f"{mode:16} accepted={row['accepted']} rejected={row['rejected']} "
          f"remaining={row['remaining']} invariant_ok={row['invariant_ok']}")


unlocked         accepted=2 rejected=0 remaining=0 invariant_ok=False
write_only       accepted=2 rejected=0 remaining=0 invariant_ok=False
whole_operation  accepted=1 rejected=1 remaining=0 invariant_ok=True


---

### 3. Verify the invariant

Starting stock must equal accepted reservations plus remaining stock. Both broken cases can leave the displayed stock at zero while accepting two reservations. A non-negative counter alone is not enough.

Repeat the controlled test to check the implementation. These repetitions are correctness checks, not a timing benchmark or a measurement of how often production races occur. Either worker may win the correct case; lock acquisition is not guaranteed to be FIFO.

In [3]:
for _ in range(25):
    for mode in ("unlocked", "write_only"):
        row = run_case(mode)
        assert row["accepted"] == 2 and row["remaining"] == 0
        assert not row["invariant_ok"]
    row = run_case("whole_operation")
    assert row["accepted"] == 1 and row["rejected"] == 1
    assert row["remaining"] == 0 and row["invariant_ok"]
print("25 repetitions per case: expected outcomes verified.")


25 repetitions per case: expected outcomes verified.


---

### 4. Release is not rollback

`with lock:` releases the lock when the block exits, including when an exception propagates. It does **not** undo completed changes. Keep critical sections short and avoid waiting for unrelated work while holding the lock.

In [4]:
lock = Lock()
state = {"value": 0}
try:
    with lock:
        state["value"] = 1
        raise RuntimeError("Demonstration failure")
except RuntimeError:
    pass

reacquired = lock.acquire(timeout=0.5)
assert reacquired
if reacquired:
    lock.release()
assert state["value"] == 1
print("Lock can be acquired again:", reacquired)
print("State after exception:", state, "(no automatic rollback)")


Lock can be acquired again: True
State after exception: {'value': 1} (no automatic rollback)


---

### 5. Save and interpret

Every access participating in this reservation protocol must follow the same lock discipline. A new lock per call provides no shared exclusion. An unrelated reader that ignores the lock is not protected. The GIL is not a substitute for protecting a multi-step application invariant.

Use the primitive that fits the coordination: `Lock` for mutual exclusion, `Event` for a signal, `Condition` to wait for a state predicate, and `Semaphore` to limit concurrent permits. A `threading.Lock` here coordinates only threads sharing this object in one process.

In [5]:
report = {
    "python": platform.python_version(), "implementation": platform.python_implementation(),
    "initial_stock": 1, "workers": 2, "cases": results,
    "repetitions_per_case": 25, "repeat_checks": "passed",
    "exception_check": {"reacquired": reacquired, "state_after": state},
    "scope": "Controlled interleaving in one process; not a race-frequency or speed benchmark.",
}
Path("results.json").write_text(json.dumps(report, indent=2))
print("Saved results.json")


Saved results.json


---

**Takeaways:** protect the complete check-and-update operation; share the same lock; use a context manager and keep the critical section focused. Test the invariant, not just the final counter.

---